# 🎯 Notebook 04 — XGBoost: Risco de Violação de OLA
## Predictfy × Locaweb — FIAP Challenge 2026

---

**Objetivo:** Treinar um classificador binário para prever, **no momento da abertura** de um incidente KPI,
se ele irá ou não violar o OLA (P2 ≤ 4h · P3 ≤ 12h).

| | |
|---|---|
| **Input** | `data/processed/incidents_features.parquet` (gerado pelo Notebook 02) |
| **Output** | `outputs/risco_ola.json` |
| **Algoritmo** | XGBoost + SMOTE + Threshold otimizado por F1 |
| **Desafio** | Desbalanceamento — usar Recall + F1 + PR-AUC (NÃO acurácia) |
| **Zero leakage** | Duração, Resolvido, Encerrado, Código de fechamento, KPI Violado? — excluídos |

---

In [ ]:
# ── 0. Setup — imports e configurações ───────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve, f1_score,
    precision_score, recall_score
)
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import shap
import json, os
from datetime import date

pd.set_option('display.max_columns', 30)
np.random.seed(42)

print('✅ Imports ok')
print(f'   XGBoost  {xgb.__version__}')
print(f'   SHAP     {shap.__version__}')


## 1. Carregar Dataset de Features

In [ ]:
# ── Caminhos ──────────────────────────────────────────────────────────────────
PARQUET_PATH = '../data/processed/incidents_features.parquet'
RAW_PATH     = '../data/raw/LW-DATASET.xlsx'

# ── Tentar carregar Parquet gerado pelo Notebook 02 ───────────────────────────
if os.path.exists(PARQUET_PATH):
    df = pd.read_parquet(PARQUET_PATH)
    print(f'✅ Parquet carregado: {df.shape}')
    print(f'   Colunas: {list(df.columns)}')

else:
    print('⚠️  incidents_features.parquet não encontrado.')
    print('   Gerando features inline a partir do dataset bruto...')
    print('   ⚡ Para resultado idêntico ao Notebook 02, execute-o primeiro.\n')

    df_raw = pd.read_excel(RAW_PATH, parse_dates=['Aberto', 'Resolvido', 'Encerrado'])
    print(f'   Dataset bruto: {df_raw.shape}')

    kpi = df_raw[df_raw['Entrou para KPI?'] == 'SIM'].copy()
    kpi = kpi[kpi['Prioridade'].isin(['2 - Alta', '3 - Média'])].copy()
    kpi = kpi.sort_values('Aberto').reset_index(drop=True)
    print(f'   Subset KPI: {kpi.shape}')

    OLA_MAP = {'2 - Alta': 14400, '3 - Média': 43200}
    kpi['ola_segundos'] = kpi['Prioridade'].map(OLA_MAP)
    kpi['target_ola']   = (kpi['Duração'] > kpi['ola_segundos']).astype(int)

    kpi['data']       = kpi['Aberto'].dt.normalize()
    kpi['hora']       = kpi['Aberto'].dt.hour
    kpi['dia_semana'] = kpi['Aberto'].dt.dayofweek
    kpi['mes']        = kpi['Aberto'].dt.month
    kpi['trimestre']  = kpi['Aberto'].dt.quarter
    kpi['dia_mes']    = kpi['Aberto'].dt.day
    kpi['semana_ano'] = kpi['Aberto'].dt.isocalendar().week.astype(int)

    kpi['is_horario_comercial'] = (
        (kpi['hora'] >= 8) & (kpi['hora'] <= 18) & (kpi['dia_semana'] < 5)
    ).astype(int)
    kpi['is_fim_de_semana'] = (kpi['dia_semana'] >= 5).astype(int)
    kpi['is_segunda_terca'] = (kpi['dia_semana'] <= 1).astype(int)

    def periodo_dia(h):
        if 0 <= h < 6:    return 0
        elif 6 <= h < 12: return 1
        elif 12 <= h < 18: return 2
        else:             return 3
    kpi['periodo_dia'] = kpi['hora'].apply(periodo_dia)

    vol = kpi.groupby('data').size().reset_index(name='vol_dia').sort_values('data')
    vol['lag_1d']      = vol['vol_dia'].shift(1)
    vol['lag_7d']      = vol['vol_dia'].shift(7)
    vol['rolling_7d']  = vol['vol_dia'].rolling(7,  min_periods=1).mean().round(2)
    vol['rolling_30d'] = vol['vol_dia'].rolling(30, min_periods=1).mean().round(2)
    kpi = kpi.merge(vol[['data','lag_1d','lag_7d','rolling_7d','rolling_30d']], on='data', how='left')

    kpi['prioridade_bin'] = (kpi['Prioridade'] == '2 - Alta').astype(int)

    for col in ['Produto', 'Categoria', 'Subcategoria', 'Grupo designado', 'Aberto por']:
        kpi[col] = kpi[col].fillna('DESCONHECIDO')

    le = LabelEncoder()
    kpi['produto_enc']      = le.fit_transform(kpi['Produto'])
    kpi['categoria_enc']    = le.fit_transform(kpi['Categoria'])
    kpi['subcategoria_enc'] = le.fit_transform(kpi['Subcategoria'])
    kpi['grupo_enc']        = le.fit_transform(kpi['Grupo designado'])
    kpi['aberto_por_enc']   = le.fit_transform(kpi['Aberto por'])

    kpi['produto_freq'] = kpi['Produto'].map(
        kpi['Produto'].value_counts(normalize=True)).round(6)
    kpi['grupo_freq']   = kpi['Grupo designado'].map(
        kpi['Grupo designado'].value_counts(normalize=True)).round(6)

    kpi['mes_sin'] = np.sin(2 * np.pi * kpi['mes'] / 12)
    kpi['mes_cos'] = np.cos(2 * np.pi * kpi['mes'] / 12)

    df = kpi.copy()
    print(f'\n✅ Features geradas inline: {df.shape}')


In [ ]:
# ── Definir features e target (sem data leakage) ─────────────────────────────
FEATURES = [
    # Temporais
    'hora', 'dia_semana', 'mes', 'trimestre', 'dia_mes', 'semana_ano',
    # Binárias / ordinais
    'is_horario_comercial', 'is_fim_de_semana', 'is_segunda_terca', 'periodo_dia',
    # Lags de volume (histórico disponível no momento da abertura)
    'lag_1d', 'lag_7d', 'rolling_7d', 'rolling_30d',
    # Prioridade
    'prioridade_bin',
    # Categóricas codificadas
    'produto_enc', 'categoria_enc', 'subcategoria_enc', 'grupo_enc', 'aberto_por_enc',
    # Frequency encoding
    'produto_freq', 'grupo_freq',
    # Cíclicas
    'mes_sin', 'mes_cos',
]
TARGET = 'target_ola'

missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f'❌ Features faltando: {missing}')
    raise ValueError('Features incompletas. Execute o Notebook 02.')
else:
    print(f'✅ Todas as {len(FEATURES)} features disponíveis')

df_model = df[FEATURES + [TARGET]].copy()
df_model[['lag_1d','lag_7d','rolling_7d','rolling_30d']] = \
    df_model[['lag_1d','lag_7d','rolling_7d','rolling_30d']].fillna(0)

assert df_model.isnull().sum().sum() == 0, '❌ Dataset com nulos!'
print(f'✅ Dataset de modelagem: {df_model.shape}')
print(f'   {len(FEATURES)} features  |  1 target  |  0 nulos')


## 2. Análise do Desbalanceamento

In [ ]:
# ── Distribuição do target ────────────────────────────────────────────────────
contagem = df_model[TARGET].value_counts().sort_index()
pct_sim  = contagem[1] / len(df_model) * 100

print('📊 Distribuição do target (target_ola):')
print(f'   NAO (0) = {contagem[0]:>7,}  ({100-pct_sim:.2f}%)')
print(f'   SIM (1) = {contagem[1]:>7,}  ({pct_sim:.2f}%)')
print(f'\n⚠️  Razão desbalanceamento ≈ 1 : {contagem[0]//contagem[1]}')
print(f'   → Nunca usar Acurácia como métrica!')
print(f'   → Métricas corretas: Recall · F1 · PR-AUC · ROC-AUC')

SCALE_POS_WEIGHT = int(contagem[0] // contagem[1])
print(f'\n   scale_pos_weight = {SCALE_POS_WEIGHT}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bars = axes[0].bar(['NAO (0)', 'SIM (1)'], contagem.values,
                   color=['#5ac8fa', '#ff2d55'], edgecolor='white', width=0.6)
axes[0].set_title('Distribuição do Target', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Nº de incidentes')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, contagem.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                 f'{val:,}\n({val/len(df_model)*100:.2f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[1].pie(contagem.values, labels=['NAO (0)', 'SIM (1)'],
            colors=['#5ac8fa', '#ff2d55'], autopct='%1.2f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Proporção do Target', fontsize=12, fontweight='bold')

plt.suptitle(f'Desbalanceamento de Classes — 1:{SCALE_POS_WEIGHT}', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 3. Split Temporal — Treino (80%) / Teste (20%)

In [ ]:
# ── Split temporal: mantendo ordem cronológica ───────────────────────────────
n_total  = len(df_model)
n_treino = int(n_total * 0.80)

X = df_model[FEATURES].values
y = df_model[TARGET].values

X_train, X_test = X[:n_treino], X[n_treino:]
y_train, y_test = y[:n_treino], y[n_treino:]

viol_train = int(y_train.sum())
viol_test  = int(y_test.sum())

print('📊 Split temporal (80% / 20%):')
print(f'   Treino: {len(X_train):>7,} amostras  |  {viol_train} violações ({viol_train/len(y_train)*100:.2f}%)')
print(f'   Teste:  {len(X_test):>7,} amostras  |  {viol_test} violações ({viol_test/len(y_test)*100:.2f}%)')
print()
print('💡 Nota sobre P2:')
print('   P2 só entra no subset KPI a partir de 2025 → dados históricos de P2 são limitados.')
print('   O modelo aprende principalmente a partir de P3 e generaliza para P2.')

if viol_test < 10:
    print(f'\n⚠️  Poucos exemplos de violação no teste ({viol_test}). Métricas podem ser instáveis.')


## 4. SMOTE — Oversampling Apenas no Treino

In [ ]:
# ── Aplicar SMOTE SOMENTE no conjunto de treino ──────────────────────────────
print('🔄 Aplicando SMOTE no treino...')
print(f'   Antes:  {viol_train} violações em {len(y_train):,} amostras ({viol_train/len(y_train)*100:.2f}%)')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'   Depois: {int(y_train_sm.sum())} violações em {len(y_train_sm):,} amostras ({y_train_sm.sum()/len(y_train_sm)*100:.1f}%)')
print()
print('✅ SMOTE aplicado — classes balanceadas 50/50 no treino')
print(f'⚠️  Teste permanece inalterado: {viol_test} violações em {len(y_test):,} amostras (distribuição real)')


## 5. Treinar XGBoost — Modelo Base (scale_pos_weight)

In [ ]:
print('🧠 Treinando Modelo Base: XGBoost + scale_pos_weight...\n')

PARAMS_BASE = dict(
    n_estimators      = 300,
    max_depth         = 6,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 5,
    scale_pos_weight  = SCALE_POS_WEIGHT,
    eval_metric       = 'aucpr',
    random_state      = 42,
    n_jobs            = -1,
    tree_method       = 'hist',
)

model_base = xgb.XGBClassifier(**PARAMS_BASE)
model_base.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred_base = model_base.predict(X_test)
y_prob_base = model_base.predict_proba(X_test)[:, 1]

roc_base = roc_auc_score(y_test, y_prob_base)
pr_base  = average_precision_score(y_test, y_prob_base)

print('📊 Modelo Base — Métricas no Teste:')
print(f'   ROC-AUC : {roc_base:.4f}')
print(f'   PR-AUC  : {pr_base:.4f}   ← principal para dados desbalanceados')
print()
print(classification_report(y_test, y_pred_base, target_names=['NAO (0)', 'SIM (1)']))


## 6. Treinar XGBoost — Modelo com SMOTE

In [ ]:
print('🧠 Treinando Modelo SMOTE: XGBoost sem scale_pos_weight...\n')

PARAMS_SMOTE = dict(
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 3,
    eval_metric      = 'aucpr',
    random_state     = 42,
    n_jobs           = -1,
    tree_method      = 'hist',
)

model_smote = xgb.XGBClassifier(**PARAMS_SMOTE)
model_smote.fit(X_train_sm, y_train_sm, eval_set=[(X_test, y_test)], verbose=False)

y_pred_smote = model_smote.predict(X_test)
y_prob_smote = model_smote.predict_proba(X_test)[:, 1]

roc_smote = roc_auc_score(y_test, y_prob_smote)
pr_smote  = average_precision_score(y_test, y_prob_smote)

print('📊 Modelo SMOTE — Métricas no Teste:')
print(f'   ROC-AUC : {roc_smote:.4f}')
print(f'   PR-AUC  : {pr_smote:.4f}')
print()
print(classification_report(y_test, y_pred_smote, target_names=['NAO (0)', 'SIM (1)']))


## 7. Comparação e Escolha do Melhor Modelo

In [ ]:
# ── Comparar curvas PR e ROC ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for nome, y_prob, cor in [
    ('Base (scale_pos_weight)', y_prob_base,  '#1a73e8'),
    ('SMOTE',                   y_prob_smote, '#ff9f0a'),
]:
    p, r, _ = precision_recall_curve(y_test, y_prob)
    auc_v   = average_precision_score(y_test, y_prob)
    axes[0].plot(r, p, label=f'{nome} (PR-AUC={auc_v:.3f})', color=cor, lw=2)

    fp_, tp_, _ = roc_curve(y_test, y_prob)
    rauc        = roc_auc_score(y_test, y_prob)
    axes[1].plot(fp_, tp_, label=f'{nome} (ROC={rauc:.3f})', color=cor, lw=2)

axes[0].axhline(y=y_test.mean(), color='gray', ls='--',
                label=f'Baseline ({y_test.mean():.4f})', lw=1)
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Curva Precision-Recall', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].plot([0,1],[0,1],'k--',label='Aleatório', lw=1)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR (Recall)')
axes[1].set_title('Curva ROC', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle('Comparação: Modelo Base vs SMOTE', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

if pr_smote >= pr_base:
    melhor_nome, model_final, y_prob_final = 'SMOTE', model_smote, y_prob_smote
    best_pr = pr_smote
else:
    melhor_nome, model_final, y_prob_final = 'Base', model_base, y_prob_base
    best_pr = pr_base

print(f'✅ Melhor modelo: {melhor_nome}')
print(f'   PR-AUC Base  = {pr_base:.4f}')
print(f'   PR-AUC SMOTE = {pr_smote:.4f}')
print(f'   → Usando {melhor_nome} para etapas seguintes')


## 7.5. Validação Cruzada — Robustez Estatística

In [ ]:
# ── Cross-Validation com StratifiedKFold ─────────────────────────────────────
# Por que CV? O test set único pode ter variância alta.
# StratifiedKFold: mantém proporção de classes em cada fold (crítico aqui).
# Usamos o melhor conjunto de treino (com ou sem SMOTE) com 5 folds.

print('🔄 Validação Cruzada — 5-Fold Estratificado...')
print(f'   Modelo: {melhor_nome}  |  Usando conjunto de treino correspondente\n')

if melhor_nome == 'SMOTE':
    X_cv, y_cv = X_train_sm, y_train_sm
    params_cv  = PARAMS_SMOTE
else:
    X_cv, y_cv = X_train, y_train
    params_cv  = PARAMS_BASE

cv_splitter  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_cv     = xgb.XGBClassifier(**params_cv)
resultados_cv = cross_validate(
    model_cv, X_cv, y_cv,
    cv=cv_splitter,
    scoring={'roc_auc': 'roc_auc', 'average_precision': 'average_precision'},
    return_train_score=False,
    n_jobs=-1,
)

roc_vals  = resultados_cv['test_roc_auc']
pr_vals   = resultados_cv['test_average_precision']

print(f'   {"Fold":<6} {"ROC-AUC":>10} {"PR-AUC":>10}')
print('   ' + '-'*30)
for i, (r, p) in enumerate(zip(roc_vals, pr_vals), 1):
    print(f'   {i:<6} {r:>10.4f} {p:>10.4f}')
print('   ' + '-'*30)
print(f'   {"Média":<6} {roc_vals.mean():>10.4f} {pr_vals.mean():>10.4f}')
print(f'   {"±std":<6} {roc_vals.std():>10.4f} {pr_vals.std():>10.4f}')

roc_test = roc_auc_score(y_test, y_prob_final)
pr_test  = average_precision_score(y_test, y_prob_final)
diff_roc = abs(roc_test - roc_vals.mean())
diff_pr  = abs(pr_test  - pr_vals.mean())

print()
print(f'📊 Comparação CV vs Test set:')
print(f'   ROC-AUC  CV={roc_vals.mean():.4f} ± {roc_vals.std():.4f}  |  Test={roc_test:.4f}  |  Δ={diff_roc:.4f}')
print(f'   PR-AUC   CV={pr_vals.mean():.4f} ± {pr_vals.std():.4f}  |  Test={pr_test:.4f}  |  Δ={diff_pr:.4f}')

if diff_roc < 0.05 and diff_pr < 0.05:
    print()
    print('✅ Resultado no test set é consistente com CV — sem overfitting detectado')
else:
    print()
    print('⚠️  Divergência entre test set e CV — pode haver overfitting ao test set')


## 8. Otimização do Threshold de Decisão

In [ ]:
# ── Encontrar threshold que maximiza F1 para a classe positiva ───────────────
precision_arr, recall_arr, thresholds = precision_recall_curve(y_test, y_prob_final)

f1_arr = (2 * precision_arr[:-1] * recall_arr[:-1] /
          (precision_arr[:-1] + recall_arr[:-1] + 1e-9))

best_idx      = int(np.argmax(f1_arr))
THRESHOLD_OTM = float(thresholds[best_idx])
best_prec     = float(precision_arr[best_idx])
best_rec      = float(recall_arr[best_idx])
best_f1       = float(f1_arr[best_idx])

print(f'🎯 Threshold otimizado por máximo F1:')
print(f'   Threshold  : {THRESHOLD_OTM:.4f}  (padrão = 0.5000)')
print(f'   Precision  : {best_prec:.4f}')
print(f'   Recall     : {best_rec:.4f}  ← captura de violações reais')
print(f'   F1-Score   : {best_f1:.4f}')

y_pred_otm = (y_prob_final >= THRESHOLD_OTM).astype(int)
print(f'\n📊 Com threshold {THRESHOLD_OTM:.4f}:')
print(f'   Alertas gerados: {y_pred_otm.sum():,}  ({y_pred_otm.sum()/len(y_pred_otm)*100:.1f}% dos casos)')
print()
print(classification_report(y_test, y_pred_otm, target_names=['NAO (0)', 'SIM (1)']))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(thresholds, f1_arr, color='#1a73e8', lw=2)
axes[0].axvline(THRESHOLD_OTM, color='#ff2d55', ls='--',
                label=f'Ótimo = {THRESHOLD_OTM:.3f}  F1={best_f1:.3f}')
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('F1-Score (classe SIM)')
axes[0].set_title('F1 por Threshold', fontsize=12, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(thresholds, precision_arr[:-1], label='Precision', color='#30d158', lw=2)
axes[1].plot(thresholds, recall_arr[:-1],    label='Recall',    color='#ff9f0a', lw=2)
axes[1].axvline(THRESHOLD_OTM, color='#ff2d55', ls='--', label=f'T = {THRESHOLD_OTM:.3f}')
axes[1].set_xlabel('Threshold')
axes[1].set_title('Precision vs Recall por Threshold', fontsize=12, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Otimização do Threshold de Decisão', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

cm = confusion_matrix(y_test, y_pred_otm)
tn_v, fp_v, fn_v, tp_v = cm.ravel()
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NAO (0)', 'SIM (1)'],
            yticklabels=['NAO (0)', 'SIM (1)'], ax=ax)
ax.set_xlabel('Predito', fontsize=11); ax.set_ylabel('Real', fontsize=11)
ax.set_title(f'Matriz de Confusão — Threshold {THRESHOLD_OTM:.3f}', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'📊 TP (violações detectadas):  {tp_v}   ← capturadas')
print(f'   FN (violações perdidas):    {fn_v}   ← custo operacional alto')
print(f'   FP (falsos alarmes):        {fp_v}')
print(f'   TN (OK confirmados):        {tn_v}')


## 8.5. Trade-off Recall × Precision — Perspectiva Operacional

In [ ]:
# ── Trade-off Recall × Precision: tabela de opções para a equipe ops ────────
# O threshold F1-ótimo não é necessariamente o melhor para toda situação:
#   • Alta urgência (ex: P2 perto do OLA) → prefere recall alto, aceita mais FP
#   • Monitoramento geral                  → F1 balanceado (threshold atual)
# Esta seção apresenta as alternativas para que o time ops escolha.

print('🎯 Trade-off Recall × Precision — Opções de Threshold:\n')
print(f'   {"Threshold":>10}  {"Recall":>7}  {"Precision":>10}  {"F1":>7}  {"Alertas":>8}  {"FP":>6}  Observação')
print('   ' + '-'*78)

thr_50_encontrado  = False
thr_70_encontrado  = False
THRESHOLD_R50      = None
THRESHOLD_R70      = None

for thr in np.arange(0.10, 0.85, 0.05):
    thr = round(thr, 2)
    pred_t = (y_prob_final >= thr).astype(int)
    rec_t  = recall_score(y_test,    pred_t, zero_division=0)
    prec_t = precision_score(y_test, pred_t, zero_division=0)
    f1_t   = 2*prec_t*rec_t / (prec_t + rec_t + 1e-9)
    al_t   = int(pred_t.sum())
    fp_t   = int(((pred_t == 1) & (y_test == 0)).sum())

    obs = ''
    if abs(thr - round(THRESHOLD_OTM, 2)) < 0.025:
        obs = '◀ F1 máximo (recomendado geral)'
    elif not thr_70_encontrado and rec_t >= 0.70:
        obs = '◀ Recall ≥ 70% — escalonamento crítico'
        THRESHOLD_R70 = thr
        thr_70_encontrado = True
    elif not thr_50_encontrado and rec_t >= 0.50:
        obs = '◀ Recall ≥ 50%'
        THRESHOLD_R50 = thr
        thr_50_encontrado = True

    print(f'   {thr:>10.2f}  {rec_t:>7.3f}  {prec_t:>10.3f}  {f1_t:>7.3f}  {al_t:>8,}  {fp_t:>6}  {obs}')

print()
print('💡 Interpretação:')
print('   → Threshold baixo  = mais alertas, mais recall, mais falsos alarmes (ruído)')
print('   → Threshold alto   = menos alertas, mais precisos, perde violações reais')
print(f'   → {THRESHOLD_OTM:.4f} maximiza F1 — ideal para monitoramento preventivo geral')
if THRESHOLD_R70:
    pred70 = (y_prob_final >= THRESHOLD_R70).astype(int)
    fp70 = int(((pred70 == 1) & (y_test == 0)).sum())
    rec70 = recall_score(y_test, pred70)
    print(f'   → {THRESHOLD_R70:.2f}   garante recall ~{rec70:.0%} — ideal para incidentes críticos (aceita {fp70} FP)')


## 9. Feature Importance — SHAP

In [ ]:
print('🔍 Calculando SHAP values...')
N_SHAP = min(2000, len(X_test))
idx    = np.random.choice(len(X_test), N_SHAP, replace=False)
X_samp = X_test[idx]

explainer   = shap.TreeExplainer(model_final)
shap_values = explainer.shap_values(X_samp)
print(f'✅ SHAP calculado para {N_SHAP} amostras')

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_samp, feature_names=FEATURES, show=False, max_display=15)
plt.title('SHAP — Importância e Direção das Features', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

shap_abs = np.abs(shap_values).mean(axis=0)
shap_df  = pd.DataFrame({'feature': FEATURES, 'shap_mean_abs': shap_abs})
shap_df  = shap_df.sort_values('shap_mean_abs', ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(shap_df['feature'], shap_df['shap_mean_abs'], color='#1a73e8', edgecolor='white')
ax.set_xlabel('Importância média |SHAP|', fontsize=11)
ax.set_title('Feature Importance — XGBoost OLA Risk', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

top10 = shap_df.sort_values('shap_mean_abs', ascending=False).head(10)
print('\n🎯 Top 10 features por importância SHAP:')
for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(f'   {i:>2}. {row["feature"]:<25}  |SHAP| = {row["shap_mean_abs"]:.5f}')


## 10. Análise de Risco por Segmento

In [ ]:
df_test           = df_model.iloc[n_treino:].copy()
df_test['prob']   = y_prob_final
df_test['alerta'] = y_pred_otm
df_test['real']   = y_test

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

risco_prio = df_test.groupby('prioridade_bin')['prob'].mean()
risco_prio.index = ['P3 (Média)', 'P2 (Alta)']
bars = axes[0].bar(risco_prio.index, risco_prio.values,
                   color=['#5ac8fa', '#ff2d55'], edgecolor='white', width=0.5)
axes[0].set_title('Probabilidade Média por Prioridade', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Prob. média de violação')
axes[0].set_ylim(0, min(1.0, risco_prio.max() * 1.5))
for bar, v in zip(bars, risco_prio.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.002,
                 f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

risco_hora = df_test.groupby('hora')['prob'].mean()
axes[1].plot(risco_hora.index, risco_hora.values, color='#1a73e8', lw=2, marker='o', ms=4)
axes[1].axvspan(8, 18, alpha=0.08, color='green', label='Horário comercial')
axes[1].set_xlabel('Hora do dia'); axes[1].set_ylabel('Prob. média')
axes[1].set_title('Risco Médio por Hora', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

axes[2].hist(df_test[df_test['real']==0]['prob'], bins=50,
             alpha=0.7, color='#5ac8fa', label='Real: NAO', density=True)
axes[2].hist(df_test[df_test['real']==1]['prob'], bins=20,
             alpha=0.85, color='#ff2d55', label='Real: SIM', density=True)
axes[2].axvline(THRESHOLD_OTM, color='black', ls='--',
                label=f'Threshold = {THRESHOLD_OTM:.3f}')
axes[2].set_xlabel('Probabilidade')
axes[2].set_title('Distribuição das Probabilidades', fontsize=11, fontweight='bold')
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.suptitle('Análise de Risco por Segmento', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

LIMITES = {'baixo': (0.0, 0.2), 'medio': (0.2, THRESHOLD_OTM), 'alto': (THRESHOLD_OTM, 1.0)}
print('📊 Distribuição por categoria de risco:')
print(f'   {"Cat":<8} {"Count":>8} {"Pct":>7}  {"Violações reais":>16}')
print('   ' + '-'*46)
for cat, (lo, hi) in LIMITES.items():
    mask = (df_test['prob'] >= lo) & (df_test['prob'] < hi)
    n    = int(mask.sum())
    viol = int(df_test.loc[mask, 'real'].sum())
    print(f'   {cat:<8} {n:>8,} {n/len(df_test)*100:>6.1f}%  {viol:>16}')


### 💡 Insight: Por que P3 tem risco maior que P2?

O modelo prevê corretamente que **P3 (Média) viola OLA mais frequentemente que P2 (Alta)**.

Isso parece contra-intuitivo (P2 tem OLA mais curto — 4h vs 12h), mas faz sentido operacional:

- **P2 (Alta)** recebe atenção imediata das equipes de suporte → é resolvida rapidamente mesmo com prazo curto
- **P3 (Média)** tem menor urgência percebida → acumula na fila e frequentemente ultrapassa as 12h
- Resultado: P2 tem **menor** taxa de violação real, P3 tem **maior** → o modelo aprende esse padrão

📌 Isso é um insight de negócio valioso: a **priorização humana** (não apenas o OLA) determina o risco real.

## 11. Exportar `outputs/risco_ola.json`

In [ ]:
shap_abs     = np.abs(shap_values).mean(axis=0)
feat_imp_list = [
    {'rank': i+1, 'feature': f, 'shap_mean_abs': round(float(v), 6)}
    for i, (f, v) in enumerate(sorted(zip(FEATURES, shap_abs), key=lambda x: -x[1]))
]

risco_prio_dict = {}
for pbin, label in [(0,'P3'),(1,'P2')]:
    mask = df_test['prioridade_bin'] == pbin
    risco_prio_dict[label] = {
        'media_prob':     round(float(df_test.loc[mask,'prob'].mean()), 4),
        'pct_alto_risco': round(float((df_test.loc[mask,'prob'] >= THRESHOLD_OTM).mean()*100), 2),
        'n_incidentes':   int(mask.sum()),
        'taxa_violacao_real': round(float(df_test.loc[mask,'real'].mean()*100), 2),
    }

dist_risco = {}
for cat, (lo, hi) in LIMITES.items():
    mask = (df_test['prob'] >= lo) & (df_test['prob'] < hi)
    dist_risco[cat] = {
        'count':           int(mask.sum()),
        'pct':             round(float(mask.sum()/len(df_test)*100), 2),
        'limite_inferior': lo,
        'limite_superior': hi if hi < 1.0 else 1.0,
        'violacoes_reais': int(df_test.loc[mask,'real'].sum()),
    }

output = {
    'modelo':              'xgboost_ola_risk',
    'gerado_em':           date.today().strftime('%Y-%m-%d'),
    'versao':              'v2',
    'abordagem':           f'XGBoost + {melhor_nome} + threshold F1-ótimo + CV validado',
    'threshold_otimizado': round(THRESHOLD_OTM, 4),
    'threshold_recall_70': round(THRESHOLD_R70, 4) if THRESHOLD_R70 else None,
    'scale_pos_weight':    SCALE_POS_WEIGHT,
    'metricas': {
        'recall_violacao':    round(best_rec,  4),
        'precision_violacao': round(best_prec, 4),
        'f1_violacao':        round(best_f1,   4),
        'roc_auc':            round(roc_auc_score(y_test, y_prob_final), 4),
        'pr_auc':             round(average_precision_score(y_test, y_prob_final), 4),
        'roc_auc_cv_mean':    round(float(roc_vals.mean()), 4),
        'roc_auc_cv_std':     round(float(roc_vals.std()),  4),
        'pr_auc_cv_mean':     round(float(pr_vals.mean()),  4),
        'pr_auc_cv_std':      round(float(pr_vals.std()),   4),
        'tp': int(tp_v), 'fp': int(fp_v), 'fn': int(fn_v), 'tn': int(tn_v),
        'total_teste':          int(len(y_test)),
        'violacoes_reais':      int(y_test.sum()),
        'violacoes_capturadas': int(tp_v),
    },
    'feature_importance_shap':  feat_imp_list[:15],
    'risco_por_prioridade':     risco_prio_dict,
    'distribuicao_risco':       dist_risco,
}

os.makedirs('../outputs', exist_ok=True)
with open('../outputs/risco_ola.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print('✅ outputs/risco_ola.json exportado! (v2)')
print(f'\n📊 Resumo final:')
print(f'   Modelo:       {melhor_nome}')
print(f'   Threshold F1: {THRESHOLD_OTM:.4f}')
print(f'   Recall:       {best_rec:.4f}')
print(f'   F1:           {best_f1:.4f}')
print(f'   PR-AUC test:  {output["metricas"]["pr_auc"]:.4f}')
print(f'   PR-AUC CV:    {output["metricas"]["pr_auc_cv_mean"]:.4f} ± {output["metricas"]["pr_auc_cv_std"]:.4f}')
print(f'   ROC-AUC test: {output["metricas"]["roc_auc"]:.4f}')
print(f'   ROC-AUC CV:   {output["metricas"]["roc_auc_cv_mean"]:.4f} ± {output["metricas"]["roc_auc_cv_std"]:.4f}')


## ✅ Notebook 04 — Concluído (v2)

### Resumo
XGBoost treinado com duas estratégias de balanceamento (scale_pos_weight e SMOTE).
Validado com StratifiedKFold 5-fold para confirmar ausência de overfitting.
Threshold otimizado por F1; alternativa de recall ≥ 70% documentada para escalonamento crítico.

### Pontos críticos respeitados
- ✅ Zero data leakage — Duração, Resolvido, Encerrado excluídos
- ✅ SMOTE apenas no treino
- ✅ Métricas corretas: Recall · F1 · PR-AUC · ROC-AUC (nunca acurácia)
- ✅ Split temporal sem shuffle
- ✅ Cross-validation valida robustez estatística
- ✅ Insight documentado: P3 viola mais que P2 por priorização humana

### Output gerado
- `outputs/risco_ola.json` (v2) — inclui métricas CV e threshold alternativo recall-70%

### Próximo passo
**Notebook 05** — K-Means: Segmentação de Padrões de Incidentes